# Astronomy, Machine Learning, and Trustworthy Evaluation

This notebook is written as a guided investigation.

We will use **one astronomy dataset** and ask **two different machine learning questions**:

1. a **classification** question: can we predict whether an asteroid is potentially hazardous?
2. a **regression** question: can we predict the diameter of an asteroid?

The goal is not only to obtain a score. The goal is to learn how to decide whether that score is actually **trustworthy**.

<div class="alert alert-block alert-info">
<b>Why only one dataset?</b>
<br>
Using one dataset keeps the story coherent.
<br>
The data stay the same, but the prediction target changes.
<br>
This makes it easier to connect the slides on:
<br>
- regressor vs classifier,
<br>
- missing values,
<br>
- class imbalance,
<br>
- one-hot encoding,
<br>
- correlation,
<br>
- outliers,
<br>
- scaling,
<br>
- evaluation metrics.
</div>

<div class="alert alert-block alert-warning">
<b>How to use this notebook</b>
<br>
This notebook contains a few steps that are intentionally naive or debatable.
<br>
You are not expected to write a lot of code from scratch.
<br>
In most cases, if something is wrong, the fix should be a small change to code that already exists:
<br>
- moving a block,
<br>
- changing an argument,
<br>
- changing a metric,
<br>
- selecting different columns,
<br>
- or deciding that a step should not be done yet.
</div>

## Before we start: a few pandas ideas

We will use `pandas` throughout the notebook.

The most important ideas for today are:

- a **DataFrame** is a table,
- each **row** is one observation,
- each **column** is one variable,
- `df.head()` shows the first rows,
- `df.info()` summarizes column types,
- `df["column"]` selects one column,
- `df[["col1", "col2"]]` selects several columns,
- `df.dropna(...)` removes rows with missing values in specific columns.

You do not need to memorize everything immediately. Read the comments in the code cells as you go.

In [ ]:
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy import stats

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    mean_squared_error,
    r2_score,
)
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.linear_model import LinearRegression

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (12, 6)
# if you are missing python packages, pip install {missingpackage}

## Load the asteroid catalogue

The file `Asteroid.csv` contains one row per asteroid and several physical or orbital properties.

In [ ]:
# Read the CSV file into a pandas DataFrame.
# low_memory=False avoids mixed-type warnings when pandas inspects the file.
df = pd.read_csv("Asteroid.csv", low_memory=False)

# Show the number of rows and columns.
df.shape

In [ ]:
# Display the first 5 rows so we can see what the table looks like.
df.head()

## First look at the dataset

Before thinking about machine learning, we should understand what kind of data we have.

In [ ]:
# df.info() shows:
# - the column names,
# - how many non-missing values each column has,
# - and the inferred data type of each column.
df.info()

In [ ]:
# Count how many values are missing in each column, as a percentage.
# This will help us decide which columns are usable and which ones are almost empty.
percent_missing = (df.isnull().sum() * 100 / len(df)).round(2)

missing_value_df = pd.DataFrame(
    {"column_name": df.columns, "percent_missing": percent_missing}
).sort_values("percent_missing", ascending=False)

missing_value_df.head(15)

<div class="alert alert-block alert-info">
<b>Concept recall: missing values</b>
<br>
From the slides: missing values can hurt model quality, but deleting data also removes information.
<br>
So we should not automatically drop everything.
<br>
Good questions to ask are:
<br>
- which columns are almost empty and probably unusable?
<br>
- which columns are scientifically important enough to keep?
<br>
- should we remove rows, or should we fill missing values?
</div>

## Choosing a small set of columns for today's work

The original dataset contains many columns, but the lesson is clearer with a smaller subset.

We will keep a few columns that are useful for today's two tasks:

- `H`: absolute magnitude,
- `albedo`: reflectivity,
- `diameter`: diameter in km,
- `data_arc`: how long the object has been observed,
- `neo`: near-Earth object flag,
- `pha`: potentially hazardous asteroid flag,
- `class`: orbital class of the asteroid.

In [ ]:
# Select only the columns we want to study today.
# .copy() creates an independent DataFrame so that later changes do not affect the original one.
astro_df = df[["H", "albedo", "diameter", "data_arc", "neo", "pha", "class"]].copy()

astro_df.head()

In [ ]:
# Check again how many values are missing, now only for our reduced table.
(astro_df.isnull().mean() * 100).round(2).sort_values(ascending=False)

## Part I - Classification

Our first question is:

**Can we predict whether an asteroid is potentially hazardous?**

This is a **classification** problem because the target is a category:

- `Y`
- `N`

In [ ]:
# Count the values of the target column.
# dropna=False also counts missing values.
astro_df["pha"].value_counts(dropna=False)

In [ ]:
# Visualize the class distribution.
# This is important because class imbalance can make accuracy misleading.
sns.countplot(data=astro_df, x="pha")
plt.title("Distribution of the target class: pha")
plt.show()

<div class="alert alert-block alert-warning">
<b>Think first</b>
<br>
One class is much rarer than the other.
<br>
Revisit the slide on class imbalance.
<br>
If a classifier always predicted the majority class, would the accuracy look bad or surprisingly good?
</div>

### A simple classification table

For this first task, we remove rows with missing values in the columns we need.

This is not the only possible choice, but it keeps the example simple for a first pass.

In [ ]:
# Keep only rows where the target and some useful predictors are present.
# We keep diameter here on purpose because it is available in this table and might look useful.
clf_df = astro_df.dropna(subset=["pha", "H", "albedo", "diameter", "data_arc", "neo", "class"]).copy()

# See how many rows remain.
clf_df.shape

In [ ]:
# Show the first rows of the cleaned classification table.
clf_df.head()

## Turning categories into numbers

Most machine learning models expect numerical inputs.

The columns `neo`, `pha`, and `class` are categorical. We therefore convert them into 0/1 columns with `pd.get_dummies`.

This is the same general idea as the slide on one-hot encoding.

In [ ]:
# Convert categorical columns into binary indicator columns.
# drop_first=True removes one category from each group to avoid a fully redundant encoding.
clf_encoded = pd.get_dummies(clf_df, columns=["neo", "pha", "class"], drop_first=True)

clf_encoded.head()

<div class="alert alert-block alert-info">
<b>What happened here?</b>
<br>
For example, the original `pha` column contained `Y` and `N`.
<br>
After one-hot encoding, this becomes a binary column such as `pha_Y`.
<br>
Value `1` means "yes", value `0` means "no".
</div>

In [ ]:
# Separate predictors from the target.
# We want to predict pha_Y, so we remove it from X and keep it in y.
X = clf_encoded.drop(columns=["pha_Y"])
y = clf_encoded["pha_Y"]

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

## A deliberately naive baseline

We now train a first classifier quickly.

The goal is to obtain a first result and then ask:
**should we trust it?**

In [ ]:
# Because the classes are imbalanced, we try a simple random oversampling strategy:
# duplicate minority-class rows until both classes have the same size.
# This is not SMOTE. It is plain resampling with replacement.
balanced_df = clf_encoded.copy()

# Separate the majority and minority classes.
majority_df = balanced_df[balanced_df["pha_Y"] == 0]
minority_df = balanced_df[balanced_df["pha_Y"] == 1]

# Duplicate minority rows with replacement until both classes are equally frequent.
minority_upsampled = minority_df.sample(
    n=len(majority_df),
    replace=True,
    random_state=42,
)

# Rebuild the balanced table and shuffle the rows.
balanced_df = pd.concat([majority_df, minority_upsampled], axis=0)
balanced_df = balanced_df.sample(frac=1, random_state=42)

# Separate predictors and target again after oversampling.
X_balanced = balanced_df.drop(columns=["pha_Y"])
y_balanced = balanced_df["pha_Y"]

y_balanced.value_counts()

In [ ]:
# Split the balanced data into train and test sets.
# test_size=0.2 means we keep 20% of the data for testing.
X_train, X_test, y_train, y_test = train_test_split(
    X_balanced,
    y_balanced,
    test_size=0.2,
    random_state=42,
)

In [ ]:
# Train a Random Forest classifier.
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train, y_train)

# Use the trained model to predict the labels of the test set.
y_pred = clf.predict(X_test)

# Print basic evaluation results.
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(
    "Interpretation: this result looks almost perfect. "
    "That is a sign that the workflow may be too optimistic, not necessarily that the model is truly excellent."
)

In [ ]:
# Draw the confusion matrix.
# This is often more informative than accuracy alone.
cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion matrix for pha classification")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.show()

<div class="alert alert-block alert-warning">
<b>Red flag</b>
<br>
The result may look excellent.
<br>
But before celebrating, ask:
<br>
- at what moment was the oversampling fitted?
<br>
- was the test set still truly "unseen" when the duplicated samples were created?
<br>
- is accuracy the only metric we should discuss for an imbalanced problem?
</div>

<div class="alert alert-block alert-info">
<b>Hint</b>
<br>
You do not need to invent a new block of code here.
<br>
If you think something is wrong, the fix is probably one of these:
<br>
- move one existing block to another place,
<br>
- add one argument to an existing function,
<br>
- or pay more attention to an evaluation metric that is already printed.
</div>

### A second issue to think about

`train_test_split` has an optional argument called `stratify`.

We did not use it above.

<div class="alert alert-block alert-warning">
<b>Question</b>
<br>
In an imbalanced classification problem, what might be the advantage of splitting the data in a stratified way?
<br>
You are not asked to write a new function. Just look at the existing `train_test_split(...)` line and think about what small change could make the train and test distributions more comparable.
</div>

## Correlation

We now look at the correlation matrix.

This is mostly an **exploration** tool: it helps us see relationships in the data.

In [ ]:
# Compute correlations between numerical columns and visualize them.
# Only numerical columns are included automatically.
plt.figure(figsize=(10, 8))
sns.heatmap(clf_encoded.corr(numeric_only=True), cmap="RdBu_r", center=0)
plt.title("Correlation matrix for the encoded classification table")
plt.show()

In [ ]:
# Keep columns whose absolute correlation is above 0.2 with at least one column.
# This is a deliberately simple rule.
corr_matrix = clf_encoded.corr(numeric_only=True)
selected_columns = corr_matrix.columns[(corr_matrix.abs() > 0.2).any()]
clf_filtered = clf_encoded[selected_columns]

clf_filtered.columns.tolist()

<div class="alert alert-block alert-warning">
<b>Red flag</b>
<br>
This filtering rule may look clever, but what is it really doing?
<br>
Is it selecting variables that are correlated with the target?
<br>
Or variables that are correlated with <i>anything at all</i>?
<br>
Those are not the same idea.
</div>

<div class="alert alert-block alert-info">
<b>Connection with the slides</b>
<br>
The correlation slides distinguish between:
<br>
- correlation with the target,
<br>
- and correlation between predictors.
<br>
A useful feature selection strategy should be clear about which of these questions it is trying to answer.
</div>

## Part II - Regression

We now switch to a second machine learning task, but we keep the same astronomy dataset.

New question:

**Can we predict asteroid diameter?**

This is a **regression** problem because the target is now a continuous value.

In [ ]:
# Create a fresh copy for the regression part.
# Using a fresh table avoids confusion with previous transformations.
reg_df = astro_df.copy()

# For regression we need diameter itself, and we also keep some predictors.
reg_df = reg_df.dropna(subset=["diameter", "albedo", "data_arc"]).copy()

# Make sure diameter is stored as a numeric value.
reg_df["diameter"] = reg_df["diameter"].astype(float)

reg_df.shape

In [ ]:
reg_df.head()

## Filling missing values

The column `H` still has some missing values.

We first inspect its distribution before deciding how to fill those gaps.

In [ ]:
# Show the skewness and a histogram of H.
# A skewed distribution often suggests that the median may be a better summary than the mean.
print("Skewness of H:", reg_df["H"].skew())

reg_df["H"].hist(bins="doane")
plt.title("Distribution of H")
plt.xlabel("H")
plt.ylabel("Count")
plt.show()

In [ ]:
# Fill missing H values with the median of H.
# This is a common simple choice for skewed numerical data.
reg_df["H"] = reg_df["H"].fillna(reg_df["H"].median())

In [ ]:
# Check whether missing values remain in the reduced regression table.
reg_df.isnull().sum()

<div class="alert alert-block alert-info">
<b>Concept recall</b>
<br>
Median imputation is often reasonable, but not automatically "the best".
<br>
The important skill is not to memorize one rule.
<br>
The important skill is to justify why a choice is reasonable.
</div>

## Numerical and categorical columns

We split the table conceptually into:

- numerical columns,
- categorical columns.

This is useful because some operations, such as VIF or scaling, are mainly designed for numerical data.

In [ ]:
# Keep only numerical columns.
reg_numeric = reg_df.select_dtypes(
    include=["int16", "int32", "int64", "float16", "float32", "float64"]
).copy()

reg_numeric.head()

In [ ]:
# Keep only categorical columns.
reg_categorical = reg_df.select_dtypes(include=["object"]).copy()

reg_categorical.head()

## Multicollinearity and VIF

The code below computes a VIF-like table.

The key idea is:

- if one predictor can be predicted very well from the others,
- then it may be highly redundant.

In [ ]:
def sklearn_vif(exogs, data):
    """Compute VIF and tolerance for a list of numerical columns."""

    vif_dict = {}
    tolerance_dict = {}

    for exog in exogs:
        # Build a temporary regression problem:
        # predict one variable using all the others.
        other_columns = [col for col in exogs if col != exog]
        X_tmp = data[other_columns]
        y_tmp = data[exog]

        # R^2 close to 1 means this variable is highly explained by the others.
        r_squared = LinearRegression().fit(X_tmp, y_tmp).score(X_tmp, y_tmp)

        # VIF becomes large when 1 - R^2 becomes small.
        vif_dict[exog] = 1 / (1 - r_squared)
        tolerance_dict[exog] = 1 - r_squared

    return pd.DataFrame({"VIF": vif_dict, "Tolerance": tolerance_dict})

In [ ]:
# Compute VIF values and repeatedly remove the worst column while VIF > 5.
# This is intentionally presented in a very automatic way.
df_vif = sklearn_vif(exogs=reg_numeric.columns, data=reg_numeric).sort_values(
    by="VIF", ascending=False
)

while (df_vif["VIF"] > 5).any():
    reduced_vif = df_vif.drop(df_vif.index[0])
    reg_numeric = reg_numeric[reduced_vif.index]
    df_vif = sklearn_vif(exogs=reg_numeric.columns, data=reg_numeric).sort_values(
        by="VIF", ascending=False
    )

df_vif.head(10)

<div class="alert alert-block alert-warning">
<b>Red flag</b>
<br>
This block looks technical, but the important question is simple:
<br>
did we accidentally include the target `diameter` in the VIF elimination process?
<br>
If yes, then our preprocessing is no longer purely about the predictors.
</div>

## One-hot encoding for categorical columns

We now encode the categorical columns, just as we did earlier for classification.

In [ ]:
# Convert categorical columns into 0/1 columns.
reg_categorical_encoded = pd.get_dummies(
    reg_categorical,
    columns=["neo", "pha", "class"],
    drop_first=True,
)

reg_categorical_encoded.head()

In [ ]:
# Merge the numerical and categorical parts back together.
clean_df = pd.concat([reg_numeric, reg_categorical_encoded], axis=1)

clean_df.shape

## Outlier methods

The slides introduced several ideas for outlier handling.

Here we show a few methods. They are **not** meant to be run one after the other in a final pipeline.

The purpose is to compare them and discuss their assumptions.

### Z-score filtering

In [ ]:
# Keep only rows where all numerical z-scores are smaller than 3 in absolute value.
clean_df_numeric = clean_df.select_dtypes(include=[np.number]).dropna()

z_scores = np.abs(stats.zscore(clean_df_numeric))
filtered_zscore = clean_df[(z_scores < 3).all(axis=1)]

filtered_zscore.shape

### Local Outlier Factor

In [ ]:
# LOF marks observations that live in sparse local neighborhoods.
lof = LocalOutlierFactor(n_neighbors=20)
lof_labels = lof.fit_predict(clean_df.select_dtypes(include=[np.number]).dropna())

filtered_lof = clean_df.iloc[np.where(lof_labels != -1)[0]]
filtered_lof.shape

### Isolation Forest

In [ ]:
# Isolation Forest isolates unusual observations by random splits.
iso_forest = IsolationForest(contamination=0.05, random_state=42)
iso_labels = iso_forest.fit_predict(clean_df.select_dtypes(include=[np.number]).dropna())

filtered_iso = clean_df.iloc[np.where(iso_labels != -1)[0]]
filtered_iso.shape

<div class="alert alert-block alert-warning">
<b>Question</b>
<br>
These methods may remove unusual observations.
<br>
But in astronomy, an unusual object is not always a bad data point.
<br>
It may be scientifically interesting.
<br>
Revisit the slides and ask:
<br>
- which methods are sensitive to scale?
<br>
- should outlier filtering be learned on the full dataset, or only on the training data?
</div>

## Scaling

We now standardize numerical columns with `StandardScaler`.

This is another place where a workflow may look correct while still being debatable.

In [ ]:
# For the rest of the notebook, we keep one filtered table.
# Here we choose the Isolation Forest result simply to continue the example.
filtered_data = filtered_iso.copy()

In [ ]:
# Select all numerical columns.
numerical_columns = filtered_data.select_dtypes(include=["number"]).columns

# Create the scaler and fit it on the current numerical table.
scaler = StandardScaler()

# Transform the numerical data so that each column is centered and scaled.
scaled_numerical_data = pd.DataFrame(
    scaler.fit_transform(filtered_data[numerical_columns]),
    columns=numerical_columns,
    index=filtered_data.index,
)

# Keep boolean columns unchanged and concatenate them with the scaled numerical columns.
clean_df_scaled = pd.concat(
    [scaled_numerical_data, filtered_data.select_dtypes(include=["bool"])],
    axis=1,
)

clean_df_scaled.head()

<div class="alert alert-block alert-warning">
<b>Red flag</b>
<br>
Look carefully at which numerical columns were scaled.
<br>
If `diameter` is still inside that list, then the target itself has been scaled.
<br>
That does not always make a model invalid, but it changes the meaning of the reported MSE and can make interpretation harder.
</div>

<div class="alert alert-block alert-info">
<b>Connection with the slides</b>
<br>
Scaling is not equally important for all models.
<br>
For example, nearest-neighbour and linear methods usually care much more about scale than tree-based methods such as Random Forest.
</div>

## Train/test split for regression

In [ ]:
# Separate predictors and target.
X = clean_df_scaled.drop(columns=["diameter"])
y = clean_df_scaled["diameter"]

# Split into train and test sets.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=0,
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## Random Forest Regressor

This is our first regression model.

In [ ]:
# Train a Random Forest regressor on the training set.
rf_reg = RandomForestRegressor(random_state=42)
rf_reg.fit(X_train, y_train)

# Predict the test targets.
rf_pred = rf_reg.predict(X_test)

# Print several evaluation quantities.
print("Training score:", rf_reg.score(X_train, y_train))
print("Testing score:", rf_reg.score(X_test, y_test))
print("MSE:", mean_squared_error(y_test, rf_pred))
print("R2:", r2_score(y_test, rf_pred))
print(
    "Interpretation: these regression numbers are computed after scaling the target. "
    "They may look unusually strong, but they are harder to interpret physically."
)

<div class="alert alert-block alert-info">
<b>Concept recall: regression metrics</b>
<br>
For regression, accuracy is not the right metric.
<br>
Instead we think about values such as:
<br>
- MSE,
<br>
- RMSE,
<br>
- R²,
<br>
- and the gap between training and testing performance.
</div>

## XGBoost Regressor

We now try another regressor.

In [ ]:
# Create an XGBoost regressor.
# The line below contains a choice that should be questioned carefully.
xgb_reg = XGBRegressor(eval_metric="logloss", random_state=42)
xgb_reg.set_params(early_stopping_rounds=30)

# We use the test set as the evaluation set for early stopping.
# This is convenient, but should we trust it?
eval_set = [(X_test, y_test)]
xgb_reg.fit(X_train, y_train, eval_set=eval_set, verbose=0)

# Evaluate on the test set.
xgb_pred = xgb_reg.predict(X_test)

print("Training score:", xgb_reg.score(X_train, y_train))
print("Testing score:", xgb_reg.score(X_test, y_test))
print("MSE:", mean_squared_error(y_test, xgb_pred))
print("R2:", r2_score(y_test, xgb_pred))
print(
    "Interpretation: if this model looks better than expected, remember that "
    "the evaluation metric and early-stopping setup are intentionally questionable here."
)

<div class="alert alert-block alert-warning">
<b>Two things to notice</b>
<br>
1. `logloss` is a metric usually associated with classification, not regression.
<br>
2. We used the test set for early stopping, which makes the final test evaluation less independent.
<br>
For this notebook, the key skill is simply to <b>notice</b> those issues and explain why they matter.
</div>

## Hyperparameter search with LightGBM

Hyperparameter tuning can be useful, but only if the metric we optimize makes sense for the task.

In [ ]:
# Define a model and a small search space.
lgbm_model = LGBMRegressor(random_state=42, verbose=-1)

param_dist = {
    "n_estimators": stats.randint(50, 400),
    "learning_rate": stats.uniform(0.01, 0.25),
    "max_depth": stats.randint(3, 20),
    "num_leaves": stats.randint(10, 60),
}

# Run a randomized search.
# The scoring choice below is intentionally questionable.
random_search = RandomizedSearchCV(
    estimator=lgbm_model,
    param_distributions=param_dist,
    n_iter=10,
    scoring="top_k_accuracy",
    cv=5,
    random_state=42,
)

random_search.fit(X_train, y_train)
random_search.best_params_
print(
    "Interpretation: the search completes, but the important question is not the chosen parameters. "
    "The important question is whether the scoring rule makes sense for regression."
)

<div class="alert alert-block alert-warning">
<b>Red flag</b>
<br>
The search procedure itself is fine.
<br>
The important question is:
<br>
does `top_k_accuracy` make sense for a regression target?
<br>
Revisit the slide on regression scoring and think about which kind of scoring string would be more appropriate.
</div>

## Final reflection

Before correcting the notebook, try to answer these questions in words:

1. Which steps are mainly for exploration, and which steps should be fitted only on training data?
2. Which cells are likely to cause data leakage?
3. Which metric choices are mismatched to the machine learning task?
4. Which results might look strong while still being methodologically weak?
5. If you wanted to correct the notebook with only small edits to existing lines, where would you start?

<div class="alert alert-block alert-success">
<b>Suggested student challenge</b>
<br>
Try to improve the notebook by changing as little code as possible.
<br>
Good candidate edits are:
<br>
- move one preprocessing step,
<br>
- add `stratify=...`,
<br>
- choose a more appropriate metric,
<br>
- or decide that one transformation should be done later in the workflow.
</div>

## Solutions and corrected workflow

The sections below give the intended answers and show how the code should be corrected.

The objective is not just to "get better numbers".
The objective is to produce numbers that are methodologically defensible.

### 1. Classification: what was wrong?

The main issues in the classification part were:

1. Oversampling was done before the train/test split.
   This leaks information from the full dataset into the training workflow.
2. The split was not stratified.
   With imbalanced classes, stratification helps preserve class proportions.
3. Accuracy was shown, but class imbalance means that precision, recall and F1-score deserve explicit attention.

The smallest defensible correction is:

- split first,
- use `stratify=y`,
- oversample only the training set,
- then evaluate on the untouched test set.

Compare the next output with the earlier "almost perfect" classification result.
The interesting part is not whether the accuracy drops a little.
The interesting part is what happens to the minority-class precision, recall and F1-score.

### 2. Corrected classification code

In [ ]:
# Recreate X and y from the encoded classification table.
X = clf_encoded.drop(columns=["pha_Y"])
y = clf_encoded["pha_Y"]

# Split first, and stratify because the target is imbalanced.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# Randomly oversample only the training set.
train_df = X_train.copy()
train_df["pha_Y"] = y_train

majority_train = train_df[train_df["pha_Y"] == 0]
minority_train = train_df[train_df["pha_Y"] == 1]

minority_train_upsampled = minority_train.sample(
    n=len(majority_train),
    replace=True,
    random_state=42,
)

balanced_train_df = pd.concat([majority_train, minority_train_upsampled], axis=0)
balanced_train_df = balanced_train_df.sample(frac=1, random_state=42)

X_train_balanced = balanced_train_df.drop(columns=["pha_Y"])
y_train_balanced = balanced_train_df["pha_Y"]

clf_corrected = RandomForestClassifier(random_state=42)
clf_corrected.fit(X_train_balanced, y_train_balanced)

y_pred_corrected = clf_corrected.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_corrected))
print(classification_report(y_test, y_pred_corrected))
print(
    "Interpretation: the overall accuracy is still high, but the minority-class scores are much lower. "
    "This is a more realistic picture of model performance on an imbalanced problem."
)

### 3. Why this classification fix is better

This corrected version is better because:

- the test set remains untouched by the oversampling step,
- the split respects class proportions,
- the classification report is interpreted together with accuracy.

The important lesson is:
a slightly lower score with a clean workflow is more valuable than a very high score produced by leakage.

### 4. Regression: what was wrong?

The main issues in the regression part were:

1. VIF was computed on a table that still included the target `diameter`.
2. Outlier removal was fitted on the full dataset before the split.
3. Scaling was fitted on the full dataset before the split.
4. The target `diameter` was scaled together with the predictors.
5. `logloss` was used inside a regressor.
6. The test set was used for early stopping.
7. `top_k_accuracy` was used in a regression hyperparameter search.

The clean logic is:

- define `X` and `y`,
- split first,
- fit preprocessing only on the training set,
- keep the test set untouched until final evaluation,
- use regression metrics for regression models.

Compare the next regression outputs with the earlier ones.
In particular, compare the meaning of the MSE before and after removing target scaling from the flawed workflow.

### 5. Corrected regression code

In [ ]:
# Start again from the cleaned regression table.
reg_df_solution = astro_df.dropna(subset=["diameter", "albedo", "data_arc"]).copy()
reg_df_solution["diameter"] = reg_df_solution["diameter"].astype(float)
reg_df_solution["H"] = reg_df_solution["H"].fillna(reg_df_solution["H"].median())

# Separate target from predictors before any later preprocessing choice.
X = reg_df_solution.drop(columns=["diameter"])
y = reg_df_solution["diameter"]

# One-hot encode only the predictors.
X = pd.get_dummies(X, columns=["neo", "pha", "class"], drop_first=True)

# Split before fitting preprocessing.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=0,
)

# Fit the scaler on training data only.
numeric_cols = X_train.select_dtypes(include=["number"]).columns

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

# Train a Random Forest regressor.
rf_reg_corrected = RandomForestRegressor(random_state=42)
rf_reg_corrected.fit(X_train_scaled, y_train)

rf_pred_corrected = rf_reg_corrected.predict(X_test_scaled)

print("Training score:", rf_reg_corrected.score(X_train_scaled, y_train))
print("Testing score:", rf_reg_corrected.score(X_test_scaled, y_test))
print("MSE:", mean_squared_error(y_test, rf_pred_corrected))
print("R2:", r2_score(y_test, rf_pred_corrected))
print(
    "Interpretation: this MSE is now expressed on the original diameter scale, "
    "so it is more meaningful scientifically than the earlier scaled-target result."
)

### 6. A corrected XGBoost example

Two things need to change:

- use a regression metric such as `rmse`,
- do not use the final test set for early stopping.

If early stopping is desired, a validation set should be taken from the training data.

In [ ]:
# Create a validation split from the training data only.
X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train_scaled,
    y_train,
    test_size=0.2,
    random_state=42,
)

xgb_reg_corrected = XGBRegressor(
    eval_metric="rmse",
    random_state=42,
    early_stopping_rounds=30,
)

xgb_reg_corrected.fit(
    X_train_sub,
    y_train_sub,
    eval_set=[(X_val, y_val)],
    verbose=0,
)

xgb_pred_corrected = xgb_reg_corrected.predict(X_test_scaled)

print("Training score:", xgb_reg_corrected.score(X_train_sub, y_train_sub))
print("Testing score:", xgb_reg_corrected.score(X_test_scaled, y_test))
print("MSE:", mean_squared_error(y_test, xgb_pred_corrected))
print("R2:", r2_score(y_test, xgb_pred_corrected))
print(
    "Interpretation: once the metric and validation logic are corrected, "
    "performance may drop substantially. That does not mean the model got worse; it means the evaluation got more honest."
)

### 7. A corrected hyperparameter search

For regression, the scoring function must also be a regression score.

Reasonable examples include:

- `neg_mean_squared_error`,
- `neg_root_mean_squared_error`,
- `r2`.

In [ ]:
lgbm_model_corrected = LGBMRegressor(random_state=42, verbose=-1)

param_dist = {
    "n_estimators": stats.randint(50, 400),
    "learning_rate": stats.uniform(0.01, 0.25),
    "max_depth": stats.randint(3, 20),
    "num_leaves": stats.randint(10, 60),
}

random_search_corrected = RandomizedSearchCV(
    estimator=lgbm_model_corrected,
    param_distributions=param_dist,
    n_iter=10,
    scoring="neg_mean_squared_error",
    cv=5,
    random_state=42,
)

random_search_corrected.fit(X_train_scaled, y_train)

print("Best parameters:", random_search_corrected.best_params_)
print("Best CV score:", random_search_corrected.best_score_)
print("Equivalent CV MSE:", -random_search_corrected.best_score_)
print(
    "Interpretation: this cross-validation score uses a regression metric, "
    "so it is directly aligned with the learning task."
)

### 8. Final summary of the intended answers

The intended conceptual answers are:

- Oversampling should be fitted only on the training data.
- Stratification is useful for imbalanced classification.
- Accuracy alone is not enough for imbalanced classification.
- Correlation filtering should be explicit about whether it is targeting correlation with the target or between predictors.
- The target should not be included in predictor-only preprocessing such as VIF filtering.
- Outlier handling, scaling, and similar transformations should be fitted on training data only.
- The final test set should not be used for early stopping or hyperparameter selection.
- Regression models should be evaluated and tuned with regression metrics, not classification metrics.